# BYOC: Build, Push, Deploy e Inferência (AIDM — Grupo 5)

- Classificador-binario
    - predictor.py: Aplicação Flask para servir inferências (predições do modelo)
    - wsgi.py: Wrapper em torno do predictor para integração com o servidor WSGI
    - nginx.conf: Configuração do Nginx como front-end do serviço de inferência
    - serve: Programa de arranque do container de serving; inicia o servidor Gunicorn
    - pyproject.toml: Definição das dependências do ambiente de inferência
    - uv.lock: Lockfile das dependências (versões fixas geradas pelo uv)
    - main.py
- Dockerfile: Instruções para construção da imagem Docker do modelo (BYOC) 
- .dockerignore: Ficheiros e pastas a excluir do contexto de build do Docker

## Push da Imagem Docker para o ECR

In [5]:
%%sh

# Name of algo -> ECR
algorithm_name=aidm-grupo-5-loan-default

# Get git sha or fallback
if git rev-parse --is-inside-work-tree >/dev/null 2>&1; then
  GIT_SHA=$(git rev-parse --short HEAD)
else
  GIT_SHA=$(date +%Y%m%d%H%M%S)
fi

cd BYOC-Single-Model/container

#make serve executable
chmod +x Classificador-binario/serve

account=$(aws sts get-caller-identity --query Account --output text)

# Região
region=$(aws configure get region)
region=${region:-eu-west-1}

fullname="${account}.dkr.ecr.${region}.amazonaws.com/${algorithm_name}:${GIT_SHA}"

# If the repository doesn't exist in ECR, create it.
aws ecr describe-repositories --repository-names "${algorithm_name}" > /dev/null 2>&1

if [ $? -ne 0 ]
then
    aws ecr create-repository --repository-name "${algorithm_name}" > /dev/null
fi

# Get the login command from ECR and execute it directly
aws ecr get-login-password --region ${region}|docker login --username AWS --password-stdin ${fullname}

# Build the docker image locally with the image name and then push it to ECR
# with the full name.

docker build --no-cache --network sagemaker -t ${algorithm_name} .
docker tag ${algorithm_name} ${fullname}

docker push ${fullname}

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores



Login Succeeded


DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.



Sending build context to Docker daemon  62.98kB
Step 1/14 : FROM python:3.9
 ---> bcd3da597491
Step 2/14 : RUN apt-get -y update && apt-get install -y --no-install-recommends          nginx          ca-certificates     && rm -rf /var/lib/apt/lists/*
 ---> Running in 50d54be60fc6
Get:1 http://deb.debian.org/debian trixie InRelease [140 kB]
Get:2 http://deb.debian.org/debian trixie-updates InRelease [47.3 kB]
Get:3 http://deb.debian.org/debian-security trixie-security InRelease [43.4 kB]
Get:4 http://deb.debian.org/debian trixie/main amd64 Packages [9670 kB]
Get:5 http://deb.debian.org/debian trixie-updates/main amd64 Packages [5412 B]
Get:6 http://deb.debian.org/debian-security trixie-security/main amd64 Packages [102 kB]
Fetched 10.0 MB in 1s (8062 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
ca-certificates is already the newest version (20250419).
The following additional packages will be installed:
  iproute2 libbpf

## Setup do Cliente do SageMaker

- [SageMaker Client](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker.html#SageMaker.Client.create_model): Model, Endpoint Config, and Endpoint Creation
- [SageMaker RunTime Client](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker-runtime.html#SageMakerRuntime.Client.invoke_endpoint): Endpoint Invocation/Testing

In [6]:
import boto3
from sagemaker import get_execution_role
import pandas as pd
import json
from time import gmtime, strftime
import time
import subprocess
import os
from io import StringIO

sm_client = boto3.client(service_name='sagemaker')
runtime_sm_client = boto3.client(service_name='sagemaker-runtime')

account_id = boto3.client('sts').get_caller_identity()['Account']
region = boto3.Session().region_name

s3_bucket = 'i32419'
prefix = "ai-deployment-monitoring-grupo-5"

role = get_execution_role()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Criação do Modelo
[Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker.html#SageMaker.Client.create_model)

In [10]:
model_name = 'loan-default-model-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())

algorithm_name = "aidm-grupo-5-loan-default"

instance_type = 'ml.m4.4xlarge'

model_package_group_name = "aidm-grupo-5-loan-default"  # o mesmo do treino

resp = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    ModelApprovalStatus="Approved",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=1
)
pkgs = resp.get("ModelPackageSummaryList", [])
if not pkgs:
    raise ValueError(
        f"Não existem Model Packages Approved no grupo {model_package_group_name}. "
        "Aprova manualmente um no console e volta a correr."
    )

model_package_arn = pkgs[0]["ModelPackageArn"]
print("Creating a model based on Approved ModelPackage:", model_package_arn)

primary_container = {
    "ModelPackageName": model_package_arn
}

# Criar recurso Model no SageMaker (liga imagem ECR + artefacto em S3)
create_model_response = sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer=primary_container
)

print("Model ARN:", create_model_response['ModelArn'])

Creating a model based on Approved ModelPackage: arn:aws:sagemaker:eu-west-1:267567228900:model-package/aidm-grupo-5-loan-default/3
Model ARN: arn:aws:sagemaker:eu-west-1:267567228900:model/loan-default-model-2026-02-09-13-12-16


## Criação da configuração do Endpoint

[Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker.html#SageMaker.Client.create_endpoint_config)

In [11]:
endpoint_config_name = 'loan-default-config' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print('Endpoint config name: ' + endpoint_config_name)

data_capture_s3 = f"s3://{s3_bucket}/{prefix}/datacapture"

create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "InstanceType": instance_type,
            "InitialInstanceCount": 1,
            "InitialVariantWeight": 1,
            "ModelName": model_name,
            "VariantName": "AllTraffic",
        }
    ],
    DataCaptureConfig={
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": data_capture_s3,
        "CaptureOptions": [
            {"CaptureMode": "Input"}, # Apenas utilizamos o input pois era o foco do enúnciado: simular data drift
        ],
        "CaptureContentTypeHeader": {
            "CsvContentTypes": ["text/csv"],
            "JsonContentTypes": ["application/json"],
        },
    },
)

print("Endpoint config Arn: " + create_endpoint_config_response['EndpointConfigArn'])
print("Data capture S3:", data_capture_s3)

Endpoint config name: loan-default-config2026-02-09-13-12-20
Endpoint config Arn: arn:aws:sagemaker:eu-west-1:267567228900:endpoint-config/loan-default-config2026-02-09-13-12-20
Data capture S3: s3://i32419/ai-deployment-monitoring-grupo-5/datacapture


## Criação do Endpoint
[Documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker.html#SageMaker.Client.create_endpoint)

In [12]:
%%time

import time

endpoint_name = 'loan-default-endpoint-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print('Endpoint name: ' + endpoint_name)

# Criar endpoint em tempo-real
create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name)
print('Endpoint Arn: ' + create_endpoint_response['EndpointArn'])

resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
status = resp['EndpointStatus']
print("Endpoint Status: " + status)

print('Waiting for {} endpoint to be in service...'.format(endpoint_name))
waiter = sm_client.get_waiter('endpoint_in_service')
waiter.wait(EndpointName=endpoint_name)
print("Endpoint is InService:", endpoint_name)

Endpoint name: loan-default-endpoint-2026-02-09-13-12-28
Endpoint Arn: arn:aws:sagemaker:eu-west-1:267567228900:endpoint/loan-default-endpoint-2026-02-09-13-12-28
Endpoint Status: Creating
Waiting for loan-default-endpoint-2026-02-09-13-12-28 endpoint to be in service...
Endpoint is InService: loan-default-endpoint-2026-02-09-13-12-28
CPU times: user 40.1 ms, sys: 22.2 ms, total: 62.4 ms
Wall time: 2min 31s


## Invocação do Endpoint

[Documentation](https://boto3.amazonaws.com/v1/documentation/api/1.9.42/reference/services/sagemaker-runtime.html#SageMakerRuntime.Client.invoke_endpoint)

In [17]:
# Teste com json

#df = pd.read_csv("data/validation.csv")
#X_val = df.drop(columns=["Status"])

#sample = X_val.iloc[:5].to_dict(orient="records")  # 5 linhas
#request_body = {"instances": sample}           # ou {"input": sample}

#payload = json.dumps(request_body)

#response = runtime_sm_client.invoke_endpoint(
#    EndpointName=endpoint_name,
#    ContentType="application/json",
#    Body=payload
#)

#result = json.loads(response["Body"].read().decode("utf-8"))
#result

{'predictions': [{'probability': 0.9993889331817627, 'prediction': 1},
  {'probability': 6.200394273037091e-05, 'prediction': 0},
  {'probability': 0.00014063062553759664, 'prediction': 0},
  {'probability': 1.4132006981526501e-05, 'prediction': 0},
  {'probability': 0.9992954730987549, 'prediction': 1}],
 'threshold': 0.5}

In [21]:
# Teste com CSV
df = pd.read_csv("data/validation.csv")
X_val = df.drop(columns=["Status"])

# escolhe 5 linhas
sample_df = X_val.iloc[:5]

payload_csv = sample_df.to_csv(index=False)  # inclui header

# Invocar o endpoint
response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=payload_csv.encode("utf-8")
)

result = response["Body"].read().decode("utf-8")
result

'probability,prediction\n0.9993889331817627,1\n6.200394273037091e-05,0\n0.00014063062553759664,0\n1.4132006981526501e-05,0\n0.9992954730987549,1\n'

In [22]:
# Versão mais legível
df_result = pd.read_csv(StringIO(result))
print(df_result)

   probability  prediction
0     0.999389           1
1     0.000062           0
2     0.000141           0
3     0.000014           0
4     0.999295           1


## Apagar o Endpoint

Before leaving this exercise, it is a good practice to delete the resources created.

In [12]:
# sm_client.delete_endpoint(EndpointName=endpoint_name)
# sm_client.delete_endpoint_config(EndpointConfigName=EndpointConfigName)
# sm_client.delete_model(ModelName=ModelName)